# TP 0 — Linear vs non-linear regression with PyTorch

**IMT · Introduction to Machine Learning · Google Colab**

## Can a straight line fit a curved relationship?

We will predict a **number** (daily electricity consumption) from one input (outdoor temperature). This is **regression**, not classification.

We compare two models on exactly the **same training and test data**:

- A **linear model**: `nn.Linear(1, 1)`, which learns a straight line.
- A **non-linear model**: a small MLP with `ReLU` activations, which can learn a curve.

Both use the **same MSE loss** and **Adam optimizer**. Only the network architecture changes.

**Time:** around 15–20 minutes. All code is provided; run the cells in order and then try the easy experiments. No GPU required.

## 1. Create and visualize non-linear data

Imagine a building: in **cold** weather it needs heating, in **mild** weather it uses less energy, and in **hot** weather it needs cooling.

This produces a **U-shaped** relationship between outdoor temperature and energy consumption.

We create **synthetic, invented data**, not actual electricity measurements. Each example has:
- **Input feature `X`:** outdoor temperature (°C).
- **Target `y`:** electricity use (kWh/day).

**Question:** Why is predicting energy consumption a regression task?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

# Invented data: a U-shaped function plus a little random noise.
temperature = rng.uniform(-10, 35, size=200)
energy = 8.0 + 0.035 * (temperature - 17.0)**2 + rng.normal(0, 1.2, len(temperature))

X = temperature.reshape(-1, 1)
y = energy.reshape(-1, 1)

plt.figure(figsize=(6, 4))
plt.scatter(X[:, 0], y[:, 0], s=20, alpha=0.7)
plt.xlabel("Outdoor temperature (°C)")
plt.ylabel("Electricity consumption (kWh/day)")
plt.title("Synthetic energy use: a U-shaped relationship")
plt.show()
print("Input shape:", X.shape, "| Target shape:", y.shape)

## 2. Split into training and test data

We keep **75% of the examples for training** and **25% for testing**.

We also standardize temperatures, using a scaler fitted **only to the training data**. This helps optimization and avoids data leakage.

Both models use these same tensors so the comparison is fair.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED
)
scaler = StandardScaler().fit(X_train)  # Fit on TRAIN data only
xtr = torch.tensor(scaler.transform(X_train), dtype=torch.float32)
xte = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.float32)

print(f"Training: {len(xtr)} examples | Test: {len(xte)} examples")

## 3. Two different regression models

**Model A: linear regression.** `nn.Linear(1, 1)` computes a weight times the input plus a bias. Its graph is always a **straight line**.

**Model B: non-linear regression.** The network `1 → 16 → ReLU → 16 → ReLU → 1` combines simple linear operations with **ReLU activations**, allowing a **curved relationship**.

Both models predict **one continuous number**. No sigmoid or softmax is needed.

Both use **MSE** to measure prediction error. The key difference is **architecture**, not the loss function.

In [ ]:
linear_model = nn.Linear(1, 1)

nonlinear_model = nn.Sequential(
    nn.Linear(1, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

criterion = nn.MSELoss()

print("Linear model:", linear_model)
print("\nNon-linear MLP:", nonlinear_model)

## 4. Train both models using the SAME five steps

1. Clear old gradients.
2. Calculate predictions (**forward pass**).
3. Compute MSE loss.
4. Backpropagate gradients.
5. Update parameters using **Adam**.

We train each model for **450 epochs**, using the same learning rate (0.03), loss, and training data.

**Question:** If both models train in the same way, why might their accuracy differ?

In [ ]:
def train_model(model, epochs=450):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
    history = []

    for epoch in range(epochs):
        optimizer.zero_grad()             # 1. Clear gradients
        predictions = model(xtr)          # 2. Forward pass
        loss = criterion(predictions, ytr) # 3. MSE loss
        loss.backward()                   # 4. Backpropagation
        optimizer.step()                  # 5. Update weights
        history.append(loss.item())

    return history

linear_history = train_model(linear_model)
nonlinear_history = train_model(nonlinear_model)

plt.figure(figsize=(6, 4))
plt.plot(linear_history, label="Linear model")
plt.plot(nonlinear_history, label="Non-linear MLP")
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Training MSE (kWh², log scale)")
plt.title("Training: linear vs non-linear")
plt.legend()
plt.show()

## 5. Compare the two models on unseen data

Our evaluation metric is **mean absolute error (MAE)**: the average absolute prediction mistake, in **kWh/day**. **Lower is better.**

We report the test MAE for each model and plot their predictions on the **same scatter plot**.

**Look for:** The linear model draws a straight line; the MLP can follow the **U-shaped data**.

In [ ]:
linear_model.eval()
nonlinear_model.eval()
with torch.no_grad():
    linear_test_pred = linear_model(xte).numpy().ravel()
    nonlinear_test_pred = nonlinear_model(xte).numpy().ravel()

linear_mae = mean_absolute_error(y_test.ravel(), linear_test_pred)
nonlinear_mae = mean_absolute_error(y_test.ravel(), nonlinear_test_pred)

print(f"Linear regression — Test MAE: {linear_mae:.2f} kWh/day")
print(f"Non-linear MLP  — Test MAE: {nonlinear_mae:.2f} kWh/day")
print(f"Improvement: {linear_mae - nonlinear_mae:.2f} kWh/day")

# Create smooth predictions for temperatures between -10°C and 35°C.
grid = np.linspace(-10, 35, 250).reshape(-1, 1)
grid_tensor = torch.tensor(scaler.transform(grid), dtype=torch.float32)
with torch.no_grad():
    linear_curve = linear_model(grid_tensor).numpy().ravel()
    nonlinear_curve = nonlinear_model(grid_tensor).numpy().ravel()

plt.figure(figsize=(8, 5))
plt.scatter(X_train[:, 0], y_train[:, 0], alpha=0.35, s=18, label="Training data")
plt.scatter(X_test[:, 0], y_test[:, 0], alpha=0.7, s=25, label="Test data")
plt.plot(grid[:, 0], linear_curve, linewidth=2.5, label="Linear model")
plt.plot(grid[:, 0], nonlinear_curve, linewidth=2.5, label="Non-linear MLP")
plt.xlabel("Outdoor temperature (°C)")
plt.ylabel("Electricity consumption (kWh/day)")
plt.title("Same dataset: straight line vs learned curve")
plt.legend()
plt.show()

## 6. Make predictions for new temperatures

What would each model predict for **0°C**, **17°C**, and **30°C**?

In the synthetic scenario, consumption increases at both cold and hot temperatures.

**Question:** Which model better reflects that behaviour?

In [ ]:
new_temperatures = np.array([0.0, 17.0, 30.0]).reshape(-1, 1)
new_inputs = torch.tensor(scaler.transform(new_temperatures), dtype=torch.float32)

with torch.no_grad():
    linear_values = linear_model(new_inputs).numpy().ravel()
    nonlinear_values = nonlinear_model(new_inputs).numpy().ravel()

for temp, a, b in zip(new_temperatures[:, 0], linear_values, nonlinear_values):
    print(f"{temp:4.0f}°C | linear: {a:5.1f} kWh/day | MLP: {b:5.1f} kWh/day")

## 7. Your turn — simple experiments

**Experiment A (easy):** In Section 6, change the temperatures to `[-5, 10, 35]`. Compare the two models.

**Experiment B (easy):** In Section 3, change both hidden layers from **16** neurons to **4**, then re-run Sections 3–6. What changes?

**Bonus:** Remove both `nn.ReLU()` activations from the MLP. Re-run Sections 3–6. Can a stack of linear layers *without activations* learn a U-shaped curve? Why not?

### Discussion

1. What makes this task regression rather than classification?
2. Why can't `nn.Linear(1, 1)` learn the U-shaped trend?
3. What do ReLU activations add?
4. Why evaluate on a test set not used for training?

**Takeaway:** Regression predicts a number, and the mapping can be **linear or non-linear**. Here the MLP fits the curved data much better, even though the loss and optimizer are the same.

**Next:** We will use a similar network for binary classification, where the output is interpreted differently and we change the loss.